In [1]:
from PIL import Image
import torch
from torchvision import transforms
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from model import ReportGenerator
from fpdf import FPDF  # For PDF report



c:\Users\aceadmin\Desktop\CSCN8010\venv\pytorch_cpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class SingleImageDataset(Dataset):
    def __init__(self, image_path, tokenizer, transform=None):
        self.image_path = image_path
        self.tokenizer = tokenizer
        self.transform = transform

    def __len__(self):
        return 1  # Only one image for inference

    def __getitem__(self, idx):
        # Open image file and ensure it is RGB (3 channels)
        img = Image.open(self.image_path).convert('RGB')  # Convert grayscale to RGB
        
        if self.transform:
            img = self.transform(img)
        
        # Return the image
        return {'img': img}


In [3]:

def generate_pdf_report(findings, impression, filename="report.pdf"):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    pdf.set_font('Arial', 'B', 16)
    pdf.cell(200, 10, txt="Radiology Report", ln=True, align='C')

    pdf.ln(10)  # Line break
    pdf.set_font('Arial', '', 12)
    pdf.multi_cell(0, 10, f"Findings:\n{findings}")
    pdf.ln(5)
    pdf.multi_cell(0, 10, f"Impression:\n{impression}")

    pdf.output(filename)


In [4]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained('t5-small')

# Define the transform to convert image to tensor
transform = transforms.Compose([transforms.ToTensor()])


In [5]:
# Path to your test image
test_image_path = "./images/images_normalized/2_IM-0652-1001.dcm.png"  # Replace with actual path

# Create the dataset and dataloader
ds = SingleImageDataset(test_image_path, tokenizer, transform=transform)
dl = DataLoader(ds, batch_size=1, shuffle=False)



In [6]:

model = ReportGenerator(embed_dim=512, decoder_name='t5-small', num_labels=1).to(device)

# Load pretrained weights
state_dict = torch.load('./best_model.pt', map_location=device)
state_dict = {k: v for k, v in state_dict.items() if not k.startswith('classifier.')}
model.load_state_dict(state_dict, strict=False)
model.eval()



ReportGenerator(
  (img_encoder): DualImageEncoder(
    (encoder_frontal): ResNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU(inplace=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (1): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=Fals

In [7]:
with torch.no_grad():
    for batch in dl:
        img = batch['img'].to(device)

        # Provide decoder_input_ids manually
        decoder_input_ids = torch.tensor([[tokenizer.pad_token_id]]).to(device)

        # Generate findings (using the same image for frontal and lateral)
        outputs_f, _ = model(img, img, decoder_input_ids=decoder_input_ids)
        pred_f_ids = torch.argmax(outputs_f.logits, dim=-1)
        pred_f = tokenizer.decode(pred_f_ids[0], skip_special_tokens=True)

        # Generate impression
        outputs_i, _ = model(img, img, decoder_input_ids=decoder_input_ids)
        pred_i_ids = torch.argmax(outputs_i.logits, dim=-1)
        pred_i = tokenizer.decode(pred_i_ids[0], skip_special_tokens=True)

        # Save the PDF report with the results
        generate_pdf_report(pred_f, pred_i, filename="radiology_report.pdf")
        print(f"Report saved as radiology_report.pdf")

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Report saved as radiology_report.pdf


In [8]:
pred_f

''